In [2]:
import pandas as pd

train_data = pd.read_csv("frames_train_with_features.csv")

train_data.head(10)

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
train_data.shape

(5251, 8)

In [ ]:
train_data["class_id"].unique()

array([1], dtype=int64)

In [ ]:
type(train_data["features"][0])

str

In [ ]:
train_data["features"][0]

'[[0.15145960450172424, 0.13233061134815216, 0.09439558535814285, 0.08494366705417633, 0.18738994002342224, 0.08354730159044266, 0.22079601883888245, 0.23736044764518738, 0.16561025381088257, 0.24055004119873047, 0.24055004119873047, 0.24055004119873047, 0.16898904740810394, 0.24055004119873047, 0.24055004119873047, 0.24055004119873047, 0.24055004119873047, 0.24055004119873047, 0.09399797767400742, 0.03679373115301132, 0.041571229696273804, 0.02565697766840458, 0.043275315314531326, 0.00970498751848936, 0.11169174313545227, 0.10594672709703445, 0.1199258491396904, 0.17348426580429077, 0.11043428629636765, 0.04122500121593475, 0.0894777849316597, 0.22040729224681854, 0.08745088428258896, 0.23047016561031342, 0.1754997819662094, 0.17368680238723755, 0.11640357226133347, 0.13263468444347382, 0.10769715905189514, 0.09494577348232269, 0.16318975389003754, 0.1655699610710144, 0.1330503523349762, 0.19249889254570007, 0.17237862944602966, 0.1830761581659317, 0.15473881363868713, 0.154518187046

In [ ]:
# Features np arrayin string olarak tutulduğu bir column olduğundan onu tekrar np arrayine çevireceğiz.
import ast

def parse_feature(features):
    features = features.replace("array(","")
    features = features.replace(")","")

    return ast.literal_eval(features)

train_data["features"] = train_data["features"].apply(parse_feature)

#train_data["features"] = train_data["features"].replace("array(","")
#train_data["features"] = train_data["features"].replace(")","")
#train_data["features"] = ast.literal_eval(train_data["features"])


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x0000017B978D6C10>>
Traceback (most recent call last):
  File "c:\Users\Talha\miniconda3\envs\aligner\Lib\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Dont Run above this code it is depricated

Trying numpy arrays

In [3]:
import numpy as np
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")

In [4]:
print(X_train.shape)
print(y_train.shape)

(47259, 7560)
(47259,)


In [5]:
X_val = np.load("X_val.npy")
y_val = np.load("y_val.npy")

X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

In [6]:
# Checking for class imbalance
unique, counts = np.unique(y_train,return_counts=True)

for clss,count in zip(unique,counts):
    print(f"Class {clss}:  {count}")

Class 0:  31999
Class 1:  15260


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)
X_test = scaler.fit_transform(X_test)

In [8]:
import joblib

joblib.dump(scaler,"scaler.pkl")

['scaler.pkl']

In [ ]:
import torch
from torch.utils.data import Dataset

class HOGDataset(Dataset):

    def __init__(self,X,y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [11]:
from torch.utils.data import DataLoader

train_dataset = HOGDataset(X_train,y_train)
val_dataset = HOGDataset(X_val,y_val)

# Batch size will be changed as hyperparamtere later
train_load = DataLoader(train_dataset,batch_size=64,shuffle=True)
val_load = DataLoader(val_dataset,batch_size=64,shuffle=False)


In [12]:
X_batch , y_batch = next(iter(train_load))

X_batch.shape

torch.Size([64, 7560])

In [14]:
y_batch.shape

torch.Size([64, 7560])

We can implement the model now